# Explorando métricas estáticas para a RQ3 (Lab02)

Material de apoio para a análise da RQ3 no Passo 4 (S03): o que cada métrica
(complexidade ciclomática, índice de manutenibilidade, LOC, duplicação)
significa, como as ferramentas do grupo calculam cada uma, e como reagem a um
mesmo comportamento implementado de forma verbosa (padrão comum em código
gerado por IA) vs. de forma enxuta.

Usa exemplos sintéticos — não os katas finais do experimento
(`lab02/katas/`) — então pode ser lido e reexecutado sem depender de nenhum
trial já coletado.

Reaproveita as mesmas funções de `lab02/scripts/metricas_estaticas.py` (#62)
que os scripts de coleta usam sobre o código real dos trials, para que a
leitura das métricas aqui seja consistente com a que vai aparecer em
`lab02/dados/metricas-estaticas.csv`.

## Requisitos para rodar

- Pacote Python `radon` (`pip install -r lab02/requirements.txt`).
- `jscpd` via `npx` (Node instalado — `npx jscpd` baixa a versão sob
  demanda na primeira execução).
- Versões travadas para o experimento, registradas em
  [`00-decisoes.md`](../../lab02/docs/00-decisoes.md): radon 6.0.1,
  jscpd 5.2.0.

Os exemplos deste notebook são escritos em arquivos temporários
(`tempfile.mkdtemp()`) só para as ferramentas de linha de comando lerem —
nada é gravado dentro do repositório.

In [1]:
import sys
import tempfile
from pathlib import Path

sys.path.insert(0, str(Path("../../lab02/scripts").resolve()))
from metricas_estaticas import (
    rodar_radon,
    rodar_jscpd,
    calcular_complexidade,
    extrair_loc,
    extrair_mi,
    extrair_duplicacao_pct,
)

DIR_EXEMPLOS = Path(tempfile.mkdtemp(prefix="lab02_exploracao_metricas_"))


def medir(nome: str, codigo: str) -> dict:
    """Escreve `codigo` num arquivo temporário e roda radon + jscpd sobre ele."""
    caminho = DIR_EXEMPLOS / f"{nome}.py"
    caminho.write_text(codigo, encoding="utf-8")

    blocos_cc = rodar_radon("cc", caminho)
    saida_raw = rodar_radon("raw", caminho)
    saida_mi = rodar_radon("mi", caminho)
    saida_jscpd = rodar_jscpd(caminho)

    cc_media, cc_max = calcular_complexidade(blocos_cc)
    return {
        "nome": nome,
        "loc": extrair_loc(saida_raw),
        "cc_media": round(cc_media, 2),
        "cc_max": cc_max,
        "cc_rank": blocos_cc[0]["rank"] if blocos_cc else None,
        "mi": round(extrair_mi(saida_mi), 2),
        "duplicacao_pct": round(extrair_duplicacao_pct(saida_jscpd), 2),
    }


def mostrar(resultado: dict) -> None:
    print(f"{resultado['nome']}:")
    for chave, valor in resultado.items():
        if chave != "nome":
            print(f"  {chave}: {valor}")

## O que é complexidade ciclomática (McCabe)

Conta o número de caminhos linearmente independentes pelo fluxo de controle
de uma função — na prática, `1 + número de pontos de decisão` (`if`,
`elif`, `for`, `while`, `and`/`or` numa condição, `except`, cada `case` de
um `match`). Quanto mais alta, mais combinações de caminho um teste
precisaria cobrir para exercitar a função inteira.

O Radon calcula a complexidade **por função/método** e converte o número
num rank de letra:

| Rank | Faixa | Interpretação do Radon |
|---|---|---|
| A | 1–5 | simples, baixo risco |
| B | 6–10 | pouco complexa |
| C | 11–20 | moderadamente complexa |
| D | 21–30 | complexa, risco elevado |
| E | 31–40 | muito complexa |
| F | 41+ | praticamente impossível de testar/entender |

Como um arquivo (`solucao.py`) pode ter mais de uma função, `metricas_estaticas.py`
(`calcular_complexidade`) resume isso em **média** e **máximo** por
trial — é por isso que `cc_media`/`cc_max` são as colunas usadas, não a
complexidade "do arquivo" como um número único.

## O que é o Índice de Manutenibilidade (MI)

Métrica composta: combina o Volume de Halstead (tamanho/vocabulário do
código), a complexidade ciclomática e o LOC numa fórmula só, normalizada
pelo Radon numa escala de 0 a 100 (mais alto = mais fácil de manter). Por
juntar três sinais diferentes, o MI tende a ser mais robusto do que olhar
qualquer um dos três isoladamente — o próprio enunciado do laboratório
recomenda essa leitura.

O Radon também dá um rank de letra para o MI, mas com faixas bem mais
permissivas que a escala "clássica" do índice (Coleman/Oman): **A** cobre
todo o intervalo de 20 a 100, **B** vai de 10 a 19, **C** fica abaixo de
10. Isso significa que arquivos com MI bem diferentes entre si (ex.: 66 e
43) podem aparecer com o mesmo rank A — o exemplo mais adiante mostra esse
caso na prática. Para comparar `com-ia` vs. `sem-ia`, o **valor numérico**
do MI é mais informativo do que o rank do Radon.

## Por que reportar LOC junto

CC e duplicação, olhadas sozinhas, podem enganar: código gerado por IA
tende a ser mais verboso para o mesmo comportamento, e mais linhas por si
só já pressionam o MI para baixo (a fórmula usa `ln(LOC)`) mesmo sem
nenhum aumento real de dificuldade de manutenção. Reportar `loc` ao lado de
`cc_media`/`cc_max`/`duplicacao_pct` é o que permite diferenciar "este
código é mais complexo" de "este código é só mais longo" — convenção já
registrada em
[`01-desenho-experimento.md`](../../lab02/docs/01-desenho-experimento.md)
(#64) como controle obrigatório da RQ3, não uma métrica testada
isoladamente.

## Como o jscpd calcula duplicação

O jscpd tokeniza o arquivo (o tokenizador é ciente da linguagem — aqui,
Python) e procura sequências de tokens que se repetem em pelo menos
`--min-lines` linhas **e** `--min-tokens` tokens ao mesmo tempo. Os valores
usados pelo grupo (`rodar_jscpd`, `metricas_estaticas.py`) são os mesmos
fixados em `00-decisoes.md`: `min-lines=3`, `min-tokens=20`. Cada trecho
repetido encontrado é um "clone"; a métrica extraída
(`extrair_duplicacao_pct`, campo `statistics.total.percentage` do relatório
JSON) é a **porcentagem de linhas do arquivo que participam de pelo menos
um clone**.

Importante: o jscpd pega cópia literal (ou quase literal, com identificadores
trocados) de blocos — ele não identifica "duas funções com texto diferente
mas o mesmo comportamento" como duplicação. Isso é uma limitação real da
métrica, não um detalhe de implementação: duplicação por copiar-e-colar
aparece; duplicação por reinventar a mesma lógica com nomes diferentes,
não.

## Exemplo — mesma função, duas implementações: "enxuta" vs. "verbosa (estilo IA)"

As duas funções abaixo calculam a mesma coisa: o total de um pedido, com um
desconto percentual por categoria do item. A versão **enxuta** usa uma
tabela de desconto (`dict`) e um laço só. A versão **verbosa** — um padrão
comum em primeiros rascunhos gerados por IA — repete o mesmo bloco de
lógica (subtotal, desconto, checagem de `None`, soma ao total) uma vez por
categoria, em vez de fatorar a parte comum.

In [2]:
ENXUTO = """
def calcular_total_pedido(itens: list[dict]) -> float:
    \"\"\"Soma o subtotal de cada item (quantidade * preco_unitario), com desconto por categoria.\"\"\"
    descontos = {"eletronico": 0.10, "alimento": 0.05, "outro": 0.0}
    total = 0.0
    for item in itens:
        desconto = descontos.get(item["categoria"], 0.0)
        subtotal = item["quantidade"] * item["preco_unitario"]
        total += subtotal * (1 - desconto)
    return round(total, 2)
"""

resultado_enxuto = medir("enxuto", ENXUTO)
mostrar(resultado_enxuto)

enxuto:
  loc: 8
  cc_media: 2.0
  cc_max: 2
  cc_rank: A
  mi: 66.59
  duplicacao_pct: 0.0


In [3]:
VERBOSO = """
def calcular_total_pedido(itens):
    total = 0.0
    for item in itens:
        if item["categoria"] == "eletronico":
            if item["quantidade"] is not None and item["preco_unitario"] is not None:
                subtotal = item["quantidade"] * item["preco_unitario"]
                desconto = subtotal * 0.10
                total_item = subtotal - desconto
                total = total + total_item
            else:
                total_item = 0
                total = total + total_item
        elif item["categoria"] == "alimento":
            if item["quantidade"] is not None and item["preco_unitario"] is not None:
                subtotal = item["quantidade"] * item["preco_unitario"]
                desconto = subtotal * 0.05
                total_item = subtotal - desconto
                total = total + total_item
            else:
                total_item = 0
                total = total + total_item
        else:
            if item["quantidade"] is not None and item["preco_unitario"] is not None:
                subtotal = item["quantidade"] * item["preco_unitario"]
                desconto = subtotal * 0.0
                total_item = subtotal - desconto
                total = total + total_item
            else:
                total_item = 0
                total = total + total_item
    return round(total, 2)
"""

resultado_verboso = medir("verboso", VERBOSO)
mostrar(resultado_verboso)

verboso:
  loc: 31
  cc_media: 10.0
  cc_max: 10
  cc_rank: B
  mi: 47.88
  duplicacao_pct: 43.75


In [4]:
print(f"{'métrica':<14}{'enxuto':>10}{'verboso':>10}")
for chave in ("loc", "cc_media", "cc_rank", "mi", "duplicacao_pct"):
    print(f"{chave:<14}{str(resultado_enxuto[chave]):>10}{str(resultado_verboso[chave]):>10}")

métrica           enxuto   verboso
loc                    8        31
cc_media             2.0      10.0
cc_rank                A         B
mi                 66.59     47.88
duplicacao_pct       0.0     43.75


**Leitura:** as duas funções fazem exatamente a mesma coisa, mas a versão
verbosa sai com bem mais linhas, complexidade ciclomática visivelmente
maior (cada categoria vira seu próprio bloco de decisão, em vez de uma
busca em tabela) e uma fração grande do arquivo marcada como duplicada pelo
jscpd — os três blocos `if`/`elif`/`else` são quase idênticos entre si. O
índice de manutenibilidade cai bastante em termos absolutos, mesmo que o
rank de letra do Radon não capture isso tão bem (próxima seção).

É exatamente esse tipo de diferença que a RQ3 quer detectar: não se a IA
"acerta" o problema, mas se o código que ela produz tende a ser
estruturalmente mais verboso/duplicado que uma solução manual equivalente.

## Exemplo — complexidade ciclomática alta isolada

Um cálculo de frete combinando região, faixa de peso, envio expresso,
fragilidade e valor declarado — um caso realista de função com muitas
regras de negócio independentes, sem nenhuma duplicação de bloco (para
isolar o efeito da complexidade do efeito de duplicação visto no exemplo
anterior).

In [5]:
COMPLEXO = """
def calcular_frete(peso, regiao, expresso, fragil, valor_declarado):
    if peso <= 0:
        return None
    if regiao == "norte":
        base = 25.0
    elif regiao == "nordeste":
        base = 20.0
    elif regiao == "sudeste":
        base = 12.0
    elif regiao == "sul":
        base = 15.0
    elif regiao == "centro-oeste":
        base = 18.0
    else:
        base = 30.0

    if peso > 30:
        base += 40
    elif peso > 20:
        base += 25
    elif peso > 10:
        base += 12
    elif peso > 5:
        base += 5

    if expresso:
        if regiao in ("norte", "nordeste"):
            base *= 2.2
        else:
            base *= 1.8
    else:
        if regiao in ("norte", "nordeste"):
            base *= 1.3

    if fragil:
        if valor_declarado and valor_declarado > 1000:
            base += 15
        else:
            base += 8

    if valor_declarado and valor_declarado > 5000:
        if fragil:
            base *= 1.15
        else:
            base *= 1.08

    return round(base, 2)
"""

resultado_complexo = medir("complexo", COMPLEXO)
mostrar(resultado_complexo)

complexo:
  loc: 42
  cc_media: 20.0
  cc_max: 20
  cc_rank: C
  mi: 43.52
  duplicacao_pct: 0.0


**Leitura:** a complexidade sobe para a faixa "moderada" (rank C) do Radon
— nada de duplicação de bloco aqui, é uma única função com muitos pontos de
decisão genuinamente diferentes. O MI cai para um valor próximo ao do
exemplo verboso anterior, mas por um motivo estrutural diferente
(complexidade real, não repetição de código) — o que reforça por que o
enunciado pede para olhar complexidade **e** duplicação **e** LOC juntos em
vez de confiar só no MI como resumo único: o mesmo MI baixo pode vir de
causas bem diferentes, e só complexidade + duplicação + LOC lado a lado
deixam isso visível.

## Tabela consolidada dos três exemplos

In [6]:
resultados = [resultado_enxuto, resultado_verboso, resultado_complexo]
colunas = ["nome", "loc", "cc_media", "cc_rank", "mi", "duplicacao_pct"]

print("".join(f"{c:<16}" for c in colunas))
for r in resultados:
    print("".join(f"{str(r[c]):<16}" for c in colunas))

nome            loc             cc_media        cc_rank         mi              duplicacao_pct  
enxuto          8               2.0             A               66.59           0.0             
verboso         31              10.0            B               47.88           43.75           
complexo        42              20.0            C               43.52           0.0             


## Como isso conecta com o resto do Lab02

- Este notebook **não gera** `lab02/dados/metricas-estaticas.csv` — isso é
  papel de `python lab02/scripts/metricas_estaticas.py --lote`, rodado
  sobre o código real dos trials depois de coletados (S02). Aqui é só
  material de leitura para interpretar essas colunas quando chegar a
  análise.
- As variáveis, hipóteses e testes que vão usar `cc_media`, `loc` e
  `duplicacao_pct` estão formalizados em
  [`02-hipoteses.md`](../../lab02/docs/02-hipoteses.md) (#65) — RQ3,
  Wilcoxon pareado bicaudal, sem direção fixada a priori.
- A limitação do jscpd (só pega cópia literal, não lógica reinventada) e a
  verbosidade da IA como possível ameaça à validade de construção estão
  registradas em
  [`03-ameacas-validade.md`](../../lab02/docs/03-ameacas-validade.md)
  (#66).
- A seleção das ferramentas (Radon + jscpd, versões travadas) está em
  [`00-decisoes.md`](../../lab02/docs/00-decisoes.md) (#59).